In [13]:
import json
import pandas as pd

filepath = r'C:\Users\jonas\Desktop\Studium\Master\SS 2025\Big Data - Analyseprojekt\wanda_50p-qwen3-4b_testresults.json'

with open(filepath, 'r', encoding='utf-8') as file:
    data = json.load(file)

for item in data[:10]:
    print(item)

# Convert the JSON data to a DataFrame
df = pd.DataFrame(data)

{'id': '65cfad2d1930410b13000014', 'type': 'list', 'exact_answer': [['hemophilia A'], ['hemophilia B']], 'ideal_answer': ['Concizumab is developed for hemophilia A and B.'], 'exact_prediction': ['hemophilia A', 'hemophilia B', 'hemophilia A with inhibitors', 'hemophilia B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'hemophilia A and B with inhibitors', 'he

In [14]:
import numpy as np
from collections import Counter

def preprocess_answers(df):
    """
    Preprocess the exact_answer and exact_prediction columns to handle mixed data types
    """
    def convert_to_lowercase_list(value):
        if isinstance(value, list):
            # Handle nested lists like [[item1], [item2]]
            if value and isinstance(value[0], list):
                # Flatten nested lists
                flat_list = []
                for sublist in value:
                    if isinstance(sublist, list):
                        flat_list.extend(sublist)
                    else:
                        flat_list.append(sublist)
                return [item.lower() if isinstance(item, str) else item for item in flat_list]
            else:
                # Handle regular lists
                return [item.lower() if isinstance(item, str) else item for item in value]
        elif isinstance(value, str):
            return value.lower()
        else:
            return value
    
    df_processed = df.copy()
    df_processed['exact_answer_processed'] = df_processed['exact_answer'].apply(convert_to_lowercase_list)
    df_processed['exact_prediction_processed'] = df_processed['exact_prediction'].apply(convert_to_lowercase_list)
    
    return df_processed

def calculate_yesno_metrics(predictions, golden_answers):
    """
    Calculate accuracy and macro-averaged F-measure for yes/no questions
    """
    # Convert to lowercase for comparison
    pred_clean = [p.lower() if isinstance(p, str) else str(p).lower() for p in predictions]
    gold_clean = [g.lower() if isinstance(g, str) else str(g).lower() for g in golden_answers]
    
    # Calculate accuracy
    correct = sum(1 for p, g in zip(pred_clean, gold_clean) if p == g)
    accuracy = correct / len(predictions) if len(predictions) > 0 else 0
    
    # Calculate macro-averaged F-measure
    # For "yes" answers
    tp_yes = sum(1 for p, g in zip(pred_clean, gold_clean) if p == 'yes' and g == 'yes')
    fp_yes = sum(1 for p, g in zip(pred_clean, gold_clean) if p == 'yes' and g == 'no')
    fn_yes = sum(1 for p, g in zip(pred_clean, gold_clean) if p == 'no' and g == 'yes')
    
    precision_yes = tp_yes / (tp_yes + fp_yes) if (tp_yes + fp_yes) > 0 else 0
    recall_yes = tp_yes / (tp_yes + fn_yes) if (tp_yes + fn_yes) > 0 else 0
    f1_yes = 2 * precision_yes * recall_yes / (precision_yes + recall_yes) if (precision_yes + recall_yes) > 0 else 0
    
    # For "no" answers
    tp_no = sum(1 for p, g in zip(pred_clean, gold_clean) if p == 'no' and g == 'no')
    fp_no = sum(1 for p, g in zip(pred_clean, gold_clean) if p == 'no' and g == 'yes')
    fn_no = sum(1 for p, g in zip(pred_clean, gold_clean) if p == 'yes' and g == 'no')
    
    precision_no = tp_no / (tp_no + fp_no) if (tp_no + fp_no) > 0 else 0
    recall_no = tp_no / (tp_no + fn_no) if (tp_no + fn_no) > 0 else 0
    f1_no = 2 * precision_no * recall_no / (precision_no + recall_no) if (precision_no + recall_no) > 0 else 0
    
    # Macro-averaged F-measure
    macro_f1 = (f1_yes + f1_no) / 2
    
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'f1_yes': f1_yes,
        'f1_no': f1_no,
        'precision_yes': precision_yes,
        'recall_yes': recall_yes,
        'precision_no': precision_no,
        'recall_no': recall_no
    }

def calculate_factoid_metrics(predictions, golden_answers):
    """
    Calculate strict accuracy, lenient accuracy, and MRR for factoid questions
    """
    strict_correct = 0
    lenient_correct = 0
    reciprocal_ranks = []
    
    for pred, gold in zip(predictions, golden_answers):
        # Ensure we have lists to work with
        if not isinstance(pred, list):
            pred = [pred] if pred is not None else []
        if not isinstance(gold, list):
            gold = [gold] if gold is not None else []
        
        # Convert to lowercase for comparison
        pred_lower = [str(p).lower() for p in pred]
        gold_lower = [str(g).lower() for g in gold]
        
        # Strict accuracy: check if first prediction matches any golden answer
        if pred_lower and any(pred_lower[0] == g for g in gold_lower):
            strict_correct += 1
        
        # Lenient accuracy: check if any prediction matches any golden answer
        if any(p in gold_lower for p in pred_lower):
            lenient_correct += 1
        
        # MRR: find the rank of the first correct answer
        rank = float('inf')
        for i, p in enumerate(pred_lower, 1):
            if any(p == g for g in gold_lower):
                rank = i
                break
        
        reciprocal_ranks.append(1/rank if rank != float('inf') else 0)
    
    n = len(predictions)
    strict_accuracy = strict_correct / n if n > 0 else 0
    lenient_accuracy = lenient_correct / n if n > 0 else 0
    mrr = sum(reciprocal_ranks) / n if n > 0 else 0
    
    return {
        'strict_accuracy': strict_accuracy,
        'lenient_accuracy': lenient_accuracy,
        'mrr': mrr
    }

def calculate_list_metrics(predictions, golden_answers):
    """
    Calculate precision, recall, and F-measure for list questions
    """
    precisions = []
    recalls = []
    f_measures = []
    
    for pred, gold in zip(predictions, golden_answers):
        # Ensure we have lists to work with
        if not isinstance(pred, list):
            pred = [pred] if pred is not None else []
        if not isinstance(gold, list):
            gold = [gold] if gold is not None else []
        
        # Convert to lowercase and remove duplicates while preserving order
        pred_set = set(str(p).lower() for p in pred)
        gold_set = set(str(g).lower() for g in gold)
        
        # Calculate TP, FP, FN
        tp = len(pred_set & gold_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)
        
        # Calculate precision, recall, F-measure
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f_measure = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        f_measures.append(f_measure)
    
    return {
        'mean_precision': np.mean(precisions) if precisions else 0,
        'mean_recall': np.mean(recalls) if recalls else 0,
        'mean_f_measure': np.mean(f_measures) if f_measures else 0,
        'individual_precisions': precisions,
        'individual_recalls': recalls,
        'individual_f_measures': f_measures
    }

def evaluate_bioasq_exact_answers(df):
    """
    Main function to evaluate BioASQ exact answers
    """
    # Preprocess the data
    df_processed = preprocess_answers(df)
    
    # Filter for exact questions only
    exact_df = df_processed[df_processed["exact_answer_processed"].isnull() == False].copy()
    
    if len(exact_df) == 0:
        print("No exact questions found in the dataset.")
        return {}
    
    print(f"Found {len(exact_df)} exact questions")
    
    # Identify question types based on the golden answers
    yesno_mask = exact_df['exact_answer_processed'].apply(
        lambda x: isinstance(x, str) and x.lower() in ['yes', 'no']
    )
    
    # For factoid questions, we need to identify them
    # Typically factoid questions have single entities as answers
    factoid_mask = exact_df['exact_answer_processed'].apply(
        lambda x: isinstance(x, list) and len(x) == 1
    ) & ~yesno_mask
    
    # List questions have multiple entities
    list_mask = exact_df['exact_answer_processed'].apply(
        lambda x: isinstance(x, list) and len(x) > 1
    )
    
    results = {}
    
    # Evaluate yes/no questions
    if yesno_mask.sum() > 0:
        yesno_df = exact_df[yesno_mask]
        yesno_results = calculate_yesno_metrics(
            yesno_df['exact_prediction_processed'].tolist(),
            yesno_df['exact_answer_processed'].tolist()
        )
        results['yesno'] = yesno_results
        print(f"\nYes/No Questions ({yesno_mask.sum()} questions):")
        print(f"  Accuracy: {yesno_results['accuracy']:.4f}")
        print(f"  Macro F1: {yesno_results['macro_f1']:.4f}")
        print(f"  F1 (Yes): {yesno_results['f1_yes']:.4f}")
        print(f"  F1 (No):  {yesno_results['f1_no']:.4f}")
    
    # Evaluate factoid questions
    if factoid_mask.sum() > 0:
        factoid_df = exact_df[factoid_mask]
        factoid_results = calculate_factoid_metrics(
            factoid_df['exact_prediction_processed'].tolist(),
            factoid_df['exact_answer_processed'].tolist()
        )
        results['factoid'] = factoid_results
        print(f"\nFactoid Questions ({factoid_mask.sum()} questions):")
        print(f"  Strict Accuracy:  {factoid_results['strict_accuracy']:.4f}")
        print(f"  Lenient Accuracy: {factoid_results['lenient_accuracy']:.4f}")
        print(f"  MRR:              {factoid_results['mrr']:.4f}")
    
    # Evaluate list questions
    if list_mask.sum() > 0:
        list_df = exact_df[list_mask]
        list_results = calculate_list_metrics(
            list_df['exact_prediction_processed'].tolist(),
            list_df['exact_answer_processed'].tolist()
        )
        results['list'] = list_results
        print(f"\nList Questions ({list_mask.sum()} questions):")
        print(f"  Mean Precision: {list_results['mean_precision']:.4f}")
        print(f"  Mean Recall:    {list_results['mean_recall']:.4f}")
        print(f"  Mean F-measure: {list_results['mean_f_measure']:.4f}")
    
    return results

# Usage:
if __name__ == "__main__":
    
    print("Evaluation:")
    results = evaluate_bioasq_exact_answers(df)

Evaluation:
Found 267 exact questions

Yes/No Questions (102 questions):
  Accuracy: 0.8333
  Macro F1: 0.8208
  F1 (Yes): 0.8682
  F1 (No):  0.7733

Factoid Questions (45 questions):
  Strict Accuracy:  0.3556
  Lenient Accuracy: 0.4444
  MRR:              0.3944

List Questions (120 questions):
  Mean Precision: 0.2385
  Mean Recall:    0.2819
  Mean F-measure: 0.2293
